In [1]:
# ==============================================================================
# Cell 1: Setup & Load Schemas from the 3 Interim Datasets
# ==============================================================================
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.parquet as pq

# Set visual formatting
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 100)

# Paths to the 3 merged interim datasets
INTERIM_DIR = "../data/interim" if os.path.exists("../data/interim") else "ml/data/interim"

PATH_2017 = os.path.join(INTERIM_DIR, "CIC-IDS2017", "cicids2017_merged_raw.parquet")
PATH_2018 = os.path.join(INTERIM_DIR, "CSE-CIC-IDS2018", "cse-cic-ids2018_merged_raw.parquet")
PATH_NB15 = os.path.join(INTERIM_DIR, "UNSW-NB15", "unsw_nb15_merged_clean.parquet")

# Fallback to CSV if Parquet is not present
if not os.path.exists(PATH_2017): PATH_2017 = os.path.join(INTERIM_DIR, "CIC-IDS2017", "cicids2017_merged_raw.csv")
if not os.path.exists(PATH_2018): PATH_2018 = os.path.join(INTERIM_DIR, "CSE-CIC-IDS2018", "cse-cic-ids2018_merged_raw.csv")
if not os.path.exists(PATH_NB15): PATH_NB15 = os.path.join(INTERIM_DIR, "UNSW-NB15", "unsw_nb15_merged_clean.csv")

def get_columns(path):
    if path.endswith(".parquet"):
        return list(pq.read_schema(path).names)
    else:
        return list(pd.read_csv(path, nrows=1).columns)

cols_2017 = get_columns(PATH_2017)
cols_2018 = get_columns(PATH_2018)
cols_nb15 = get_columns(PATH_NB15)

# Clean source_file artifact from 2017 if present
cols_2017 = [c for c in cols_2017 if c != 'source_file']

print("📦 Dataset Feature Schemas Loaded:")
print(f"  • CIC-IDS2017 : {len(cols_2017)} columns from {os.path.basename(PATH_2017)}")
print(f"  • CSE-CIC-IDS2018: {len(cols_2018)} columns from {os.path.basename(PATH_2018)}")
print(f"  • UNSW-NB15   : {len(cols_nb15)} columns from {os.path.basename(PATH_NB15)}")


ArrowInvalid: Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.

In [ ]:
# ==============================================================================
# Cell 2: Literal Column Name Set Comparisons
# ==============================================================================
set_17 = set(cols_2017)
set_18 = set(cols_2018)
set_nb = set(cols_nb15)

# Set Intersections
exact_all_three = set_17 & set_18 & set_nb
exact_17_18 = set_17 & set_18
exact_17_nb = set_17 & set_nb
exact_18_nb = set_18 & set_nb

# Unique to each
only_17 = set_17 - set_18 - set_nb
only_18 = set_18 - set_17 - set_nb
only_nb = set_nb - set_17 - set_18

summary_set_algebra = pd.DataFrame([
    {'Comparison': 'A. All Three (IDS2017 ∩ IDS2018 ∩ NB15)', 'Exact Match Count': len(exact_all_three), 'Sample Matches': list(exact_all_three)},
    {'Comparison': 'B. CIC Common (IDS2017 ∩ IDS2018)', 'Exact Match Count': len(exact_17_18), 'Sample Matches': list(exact_17_18)[:5]},
    {'Comparison': 'C. IDS2017 ∩ NB15', 'Exact Match Count': len(exact_17_nb), 'Sample Matches': list(exact_17_nb)},
    {'Comparison': 'D. IDS2018 ∩ NB15', 'Exact Match Count': len(exact_18_nb), 'Sample Matches': list(exact_18_nb)},
    {'Comparison': 'Unique to IDS2017', 'Exact Match Count': len(only_17), 'Sample Matches': list(only_17)[:3]},
    {'Comparison': 'Unique to IDS2018', 'Exact Match Count': len(only_18), 'Sample Matches': list(only_18)[:3]},
    {'Comparison': 'Unique to NB15', 'Exact Match Count': len(only_nb), 'Sample Matches': list(only_nb)[:3]}
])

print("🔍 Exact Column Name Intersections:")
display(summary_set_algebra)

print("\n⚠️ Notice: Only 'Label' matches literally across all 3 because:")
print("   • 2017 uses 'Destination Port', 2018 uses 'Dst Port', and NB15 uses 'dsport'.")
print("   • 2017 uses 'Total Fwd Packets', 2018 uses 'Tot Fwd Pkts', and NB15 uses 'Spkts'.")
print("   -> We must perform SEMANTIC FEATURE MAPPING.")


🔍 Exact Column Name Intersections:


,Comparison,Exact Match Count,Sample Matches
0,A. All Three (IDS2017 ∩ IDS2018 ∩ NB15),1,[Label]
1,B. CIC Common (IDS2017 ∩ IDS2018),28,"[Fwd URG Flags, Bwd IAT Mean, CWE Flag Count, ..."
2,C. IDS2017 ∩ NB15,1,[Label]
3,D. IDS2018 ∩ NB15,1,[Label]
4,Unique to IDS2017,51,"[Fwd Packet Length Min, Bwd Avg Bytes/Bulk, Bw..."
5,Unique to IDS2018,52,"[Fwd Header Len, FIN Flag Cnt, Bwd Pkt Len Std]"
6,Unique to NB15,48,"[proto, Stime, is_sm_ips_ports]"



⚠️ Notice: Only 'Label' matches literally across all 3 because:
   • 2017 uses 'Destination Port', 2018 uses 'Dst Port', and NB15 uses 'dsport'.
   • 2017 uses 'Total Fwd Packets', 2018 uses 'Tot Fwd Pkts', and NB15 uses 'Spkts'.
   -> We must perform SEMANTIC FEATURE MAPPING.


In [ ]:
# ==============================================================================
# Cell 3: Cross-Dataset Semantic Feature Mapping Table (Updated with Packet Rates)
# ==============================================================================
semantic_catalog = [
    # Network Identifiers & Protocols
    ("Destination Port", "Destination Port", "Dst Port", "dsport", "Exact match (Port integer 0-65535)"),
    ("Source Port", None, None, "sport", "Present in NB15 (Dropped from merged CIC to prevent leakage)"),
    ("Protocol", None, "Protocol", "proto", "Numeric in 2018 (6=TCP, 17=UDP), string in NB15 ('tcp', 'udp')"),

    # Flow Duration & Timing
    ("Flow Duration", "Flow Duration", "Flow Duration", "dur", "CIC: Microseconds (μs); NB15: Seconds (dur * 1e6)"),
    ("Inter-Arrival Time (Mean)", "Flow IAT Mean", "Flow IAT Mean", "tcprtt", "CIC: μs between packets; NB15: TCP Round Trip Time in seconds"),
    ("Forward IAT Mean", "Fwd IAT Mean", "Fwd IAT Mean", "Sintpkt", "CIC: μs; NB15: Source interpacket arrival time in ms"),
    ("Backward IAT Mean", "Bwd IAT Mean", "Bwd IAT Mean", "Dintpkt", "CIC: μs; NB15: Destination interpacket arrival time in ms"),

    # Flow Volume & Packet Counts
    ("Forward Packets", "Total Fwd Packets", "Tot Fwd Pkts", "Spkts", "Total packets sent from source to destination"),
    ("Backward Packets", "Total Backward Packets", "Tot Bwd Pkts", "Dpkts", "Total packets sent from destination to source"),
    ("Forward Bytes", "Total Length of Fwd Packets", "TotLen Fwd Pkts", "sbytes", "Total volume of payload/header bytes forward"),
    ("Backward Bytes", "Total Length of Bwd Packets", "TotLen Bwd Pkts", "dbytes", "Total volume of payload/header bytes backward"),

    # Packet Size Statistics
    ("Mean Forward Packet Size", "Fwd Packet Length Mean", "Fwd Pkt Len Mean", "smeansz", "Average forward packet size in bytes"),
    ("Mean Backward Packet Size", "Bwd Packet Length Mean", "Bwd Pkt Len Mean", "dmeansz", "Average backward packet size in bytes"),

    # Rates & Throughput (NOW ALL INCLUDED IN CORE VIA DERIVATION)
    ("Flow Transfer Rate (Bytes/s)", "Flow Bytes/s", "Flow Byts/s", "Sload", "CIC: Bytes/sec; NB15: Derived as (sbytes + dbytes)/dur or (Sload/8)"),
    ("Flow Packet Rate (Packets/s)", "Flow Packets/s", "Flow Pkts/s", "Spkts+Dpkts/dur", "CIC: Native; NB15: DERIVED as (Spkts + Dpkts) / dur"),
    ("Forward Packet Rate", "Fwd Packets/s", "Fwd Pkts/s", "Spkts/dur", "CIC: Native; NB15: DERIVED as Spkts / dur"),
    ("Backward Packet Rate", "Bwd Packets/s", "Bwd Pkts/s", "Dpkts/dur", "CIC: Native; NB15: DERIVED as Dpkts / dur"),

    # TCP State & Window Properties
    ("Forward TCP Window Size", "Init_Win_bytes_forward", "Init Fwd Win Byts", "swin", "Initial TCP window advertisement (Source)"),
    ("Backward TCP Window Size", "Init_Win_bytes_backward", "Init Bwd Win Byts", "dwin", "Initial TCP window advertisement (Destination)"),

    # TCP Flags (Present in CIC only)
    ("SYN Flag Count", "SYN Flag Count", "SYN Flag Cnt", None, "CIC only"),
    ("ACK Flag Count", "ACK Flag Count", "ACK Flag Cnt", None, "CIC only"),

    # Target & Classification
    ("Target (Binary Label)", "Label", "Label", "Label", "0 for Normal/Benign, 1 for Attack"),
    ("Attack Category (Multi-class)", "Label", "Label", "attack_cat", "Multi-class threat family")
]

comparison_rows = []
for concept, c17, c18, cnb, note in semantic_catalog:
    has_17 = (c17 is not None) and (c17 in cols_2017)
    has_18 = (c18 is not None) and (c18 in cols_2018)
    
    # Check if native or derivable in NB15
    is_derivable = cnb in ["Spkts+Dpkts/dur", "Spkts/dur", "Dpkts/dur"] or (cnb is not None and cnb in cols_nb15)
    
    mark_17 = "✓" if has_17 else "—"
    mark_18 = "✓" if has_18 else "—"
    mark_nb = "✓*" if is_derivable else "—"
    
    in_all_three = has_17 and has_18 and is_derivable
    
    comparison_rows.append({
        'Feature Concept': concept,
        'IDS2017': mark_17,
        'IDS2018': mark_18,
        'NB15': mark_nb,
        'In All 3?': '✅ Core Candidate' if in_all_three else ('CIC Pair' if (has_17 and has_18) else 'Single Dataset'),
        'IDS2017 Column': c17 if c17 else '—',
        'IDS2018 Column': c18 if c18 else '—',
        'NB15 Column': cnb if cnb else '—',
        'Alignment / Derivation Rule': note
    })

df_comparison = pd.DataFrame(comparison_rows)

print("🎯 UPDATED SEMANTIC FEATURE COMPARISON TABLE:")
print("(* indicates semantic match or derived feature)\n")
display(df_comparison[['Feature Concept', 'IDS2017', 'IDS2018', 'NB15', 'In All 3?', 'Alignment / Derivation Rule']])


🎯 UPDATED SEMANTIC FEATURE COMPARISON TABLE:
(* indicates semantic match or derived feature)



,Feature Concept,IDS2017,IDS2018,NB15,In All 3?,Alignment / Derivation Rule
0,Destination Port,✓,✓,✓*,✅ Core Candidate,Exact match (Port integer 0-65535)
1,Source Port,—,—,✓*,Single Dataset,Present in NB15 (Dropped from merged CIC to pr...
2,Protocol,—,✓,✓*,Single Dataset,"Numeric in 2018 (6=TCP, 17=UDP), string in NB1..."
3,Flow Duration,✓,✓,✓*,✅ Core Candidate,CIC: Microseconds (μs); NB15: Seconds (dur * 1e6)
4,Inter-Arrival Time (Mean),✓,✓,✓*,✅ Core Candidate,CIC: μs between packets; NB15: TCP Round Trip ...
5,Forward IAT Mean,✓,✓,✓*,✅ Core Candidate,CIC: μs; NB15: Source interpacket arrival time...
6,Backward IAT Mean,✓,✓,✓*,✅ Core Candidate,CIC: μs; NB15: Destination interpacket arrival...
7,Forward Packets,✓,✓,✓*,✅ Core Candidate,Total packets sent from source to destination
8,Backward Packets,✓,✓,✓*,✅ Core Candidate,Total packets sent from destination to source
9,Forward Bytes,✓,✓,✓*,✅ Core Candidate,Total volume of payload/header bytes forward


In [ ]:
# ==============================================================================
# Cell 3B: Validate & Compare Derived Packet Rates
# ==============================================================================
print("🔬 Sampling and comparing Flow Packet Rates across the 3 datasets...")

# Load small sample of each
s17 = pd.read_parquet(PATH_2017, columns=['Flow Packets/s']) if PATH_2017.endswith('.parquet') else pd.read_csv(PATH_2017, usecols=['Flow Packets/s'], nrows=50000)
s18 = pd.read_parquet(PATH_2018, columns=['Flow Pkts/s']) if PATH_2018.endswith('.parquet') else pd.read_csv(PATH_2018, usecols=['Flow Pkts/s'], nrows=50000)
s_nb = pd.read_parquet(PATH_NB15, columns=['Spkts', 'Dpkts', 'dur', 'Label']) if PATH_NB15.endswith('.parquet') else pd.read_csv(PATH_NB15, usecols=['Spkts', 'Dpkts', 'dur', 'Label'], nrows=50000)

# Derive packet rates in NB15 sample
dur_safe = s_nb['dur'].replace(0, 1e-6)
s_nb['Derived Flow Pkts/s'] = (s_nb['Spkts'] + s_nb['Dpkts']) / dur_safe
s_nb['Derived Fwd Pkts/s'] = s_nb['Spkts'] / dur_safe
s_nb['Derived Bwd Pkts/s'] = s_nb['Dpkts'] / dur_safe

# Summary Statistics Comparison
packet_rate_comparison = pd.DataFrame({
    'Metric': ['Median (Packets/s)', '75th Percentile', 'Max (Packets/s)'],
    'CIC-IDS2017 (Native)': [
        round(s17['Flow Packets/s'].replace([np.inf, -np.inf], np.nan).median(), 2),
        round(s17['Flow Packets/s'].replace([np.inf, -np.inf], np.nan).quantile(0.75), 2),
        round(s17['Flow Packets/s'].replace([np.inf, -np.inf], np.nan).max(), 2)
    ],
    'CSE-CIC-IDS2018 (Native)': [
        round(s18['Flow Pkts/s'].replace([np.inf, -np.inf], np.nan).median(), 2),
        round(s18['Flow Pkts/s'].replace([np.inf, -np.inf], np.nan).quantile(0.75), 2),
        round(s18['Flow Pkts/s'].replace([np.inf, -np.inf], np.nan).max(), 2)
    ],
    'UNSW-NB15 (Derived)': [
        round(s_nb['Derived Flow Pkts/s'].median(), 2),
        round(s_nb['Derived Flow Pkts/s'].quantile(0.75), 2),
        round(s_nb['Derived Flow Pkts/s'].max(), 2)
    ]
})

print("\n📊 Packet Velocity Comparison Across Datasets:")
display(packet_rate_comparison)

print("\n💡 Notice how the derived NB15 packet rate aligns with the high-velocity flow scales of the CIC datasets!")


🔬 Sampling and comparing Flow Packet Rates across the 3 datasets...

📊 Packet Velocity Comparison Across Datasets:


,Metric,CIC-IDS2017 (Native),CSE-CIC-IDS2018 (Native),UNSW-NB15 (Derived)
0,Median (Packets/s),109.48,136.12,3.696860e+03
1,75th Percentile,23255.81,5494.51,6.328140e+03
2,Max (Packets/s),4000000.00,6000000.00,1.290000e+09



💡 Notice how the derived NB15 packet rate aligns with the high-velocity flow scales of the CIC datasets!
